# Night lights for the IGAD dataset (Google Colab)

This notebook processes the NASA Black Marble night lights for every month from 2012 to 2025 and saves one file per month to your Google Drive. It replaces the night-light stage that used to run on GitHub Actions.

**Before you start (one time only):** add your NASA Earthdata token as a Colab secret. Click the key icon in the left sidebar, then **Add new secret**. Name it `NASA_EARTHDATA_TOKEN`, paste the token as the value, and turn on **Notebook access**. The token stays in Colab's secret storage and does not appear in the notebook.

**How to run it:** click **Runtime**, then **Run all**. Allow access to Google Drive when Colab asks. The whole run takes about two to three hours. Keep this browser tab open while it runs.

**If it stops** (for example, Colab disconnects), click **Run all** again. Months that are already saved in your Drive are skipped, so it continues where it stopped.

**When it finishes,** the last cell saves `igad_lights.zip` in your Drive folder `igad-nightlights` and downloads it to your computer. Attach that file in the chat with Claude.

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
SAVE = Path("/content/drive/MyDrive/igad-nightlights")   # your results are saved here
(SAVE / "lights").mkdir(parents=True, exist_ok=True)
(SAVE / "rain").mkdir(parents=True, exist_ok=True)
print("Results folder:", SAVE)

## 2. Get the project code and install the packages it needs

In [ ]:
import os, shutil, subprocess

REPO = Path("/content/igad-nightlights-conflict")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "data-build",
                    "https://github.com/landontownsend/igad-nightlights-conflict.git", str(REPO)], check=True)

# The project saves night lights in data/interim/lights and rainfall in data/interim/rain.
# Point those two folders at Google Drive, so every saved month survives a disconnect.
for name in ["lights", "rain"]:
    folder = REPO / "data" / "interim" / name
    if folder.is_symlink():
        continue
    if folder.exists():
        shutil.rmtree(folder)        # older files from GitHub, made before the lit-pixel fix
    folder.symlink_to(SAVE / name)

# Packages that Colab does not already have
subprocess.run("pip install -q exactextract blackmarblepy rioxarray rasterio pyogrio h5py",
               shell=True, check=True)
print("Ready")

## 3. Load the NASA token from Colab's secret storage

In [ ]:
from google.colab import userdata
try:
    os.environ["BLACKMARBLE_TOKEN"] = userdata.get("NASA_EARTHDATA_TOKEN")
    print("Token loaded")
except Exception:
    raise RuntimeError("Add your token as a Colab secret named NASA_EARTHDATA_TOKEN "
                       "(key icon in the left sidebar), turn on Notebook access, then run this cell again")

## 4. Process the night lights, one year at a time

Each year takes about 10 minutes. The output shows each month as it is saved. A month that NASA has not published, or that fails after three tries, is skipped and listed at the end.

In [ ]:
def run_stage(args):
    """Run one stage of the project and show its output as it goes."""
    process = subprocess.Popen(["python", "-W", "ignore", "-m"] + args, cwd=REPO,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in process.stdout:
        print(line, end="")
        lines.append(line)
    process.wait()
    return process.returncode, "".join(lines)

problems = []
for year in range(2012, 2026):
    print(f"===== {year} =====")
    code_, output = run_stage(["pipeline.lights", "--year", str(year)])
    if "rejected the Earthdata token" in output:
        raise RuntimeError("NASA rejected the token. Make a new token at urs.earthdata.nasa.gov, "
                           "update the Colab secret, and run this cell again.")
    if code_ != 0:
        problems.append(year)

saved = sorted(p.name for p in (SAVE / "lights").glob("lights_*.csv"))
print(f"\nMonths saved: {len(saved)} of 168")
if problems:
    print("Years with a problem (run this cell again to retry them):", problems)

## 5. Rainfall (optional)

The rainfall server refused GitHub's computers, so the dataset has no rainfall yet. This cell tries it from Colab. It downloads about 420 monthly files and takes about an hour. If it fails, the dataset is still built without rainfall, so you can skip it.

In [ ]:
code_, output = run_stage(["pipeline.rain"])
print(f"\nRainfall years saved: {len(list((SAVE / 'rain').glob('rain_*.csv')))} of 35")

## 6. Check the results and download them

In [ ]:
import pandas as pd

files = sorted((SAVE / "lights").glob("lights_*.csv"))
months = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print("Months saved:", len(files), "of 168")
print("Localities per month:", months.groupby(["year", "month"]).size().median())
print("Lit totals present:", "lit_sum" in months.columns)
missing = [f"{y}-{m:02d}" for y in range(2012, 2026) for m in range(1, 13)
           if not (SAVE / "lights" / f"lights_{y}_{m:02d}.csv").exists()]
print("Missing months:", len(missing), missing[:12] if missing else "")

import zipfile
archive = SAVE / "igad_lights.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted((SAVE / "lights").glob("*.csv")) + sorted((SAVE / "rain").glob("*.csv")):
        z.write(f, f.relative_to(SAVE))          # stored as lights/... and rain/...
print("Saved", archive)

from google.colab import files as colab_files
colab_files.download(str(archive))